# Project 3: GitHub Copilot with MCP

**30 min project**

Build an engineering copilot that explores real GitHub repositories using MCP, guardrails, and tracing.

**Concepts:** `HostedMCPTool`, `@input_guardrail`, `@output_guardrail`, `trace`

In [ ]:
!pip install -q openai-agents python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

## 1. GitHub Copilot with Hosted MCP

In [ ]:
from agents import Agent, HostedMCPTool, Runner

copilot = Agent(
    name="GitHub Copilot",
    instructions=(
        "You are a senior engineer. Use the DeepWiki MCP server to inspect GitHub repositories. "
        "Answer with concrete file names, concepts, and architecture details when available."
    ),
    model="gpt-4o-mini",
    tools=[
        HostedMCPTool(
            tool_config={
                "type": "mcp",
                "server_label": "deepwiki",
                "server_url": "https://mcp.deepwiki.com/mcp",
                "require_approval": "never",
            }
        )
    ],
)

result = Runner.run_sync(
    copilot,
    "In openai/openai-agents-python, what are the core primitives of the SDK?",
)

print(result.final_output)

## 2. Input Guardrail

In [ ]:
from pydantic import BaseModel
from agents import GuardrailFunctionOutput, InputGuardrailTripwireTriggered, RunContextWrapper, input_guardrail


class TopicCheck(BaseModel):
    is_engineering_topic: bool
    reasoning: str


topic_guard_agent = Agent(
    name="Engineering Topic Guard",
    instructions="Decide whether the user request is about software engineering, code, architecture, repositories, debugging, or technical documentation.",
    model="gpt-4o-mini",
    output_type=TopicCheck,
)


@input_guardrail
async def engineering_topic_guardrail(ctx: RunContextWrapper[None], agent: Agent, input: str) -> GuardrailFunctionOutput:
    result = await Runner.run(topic_guard_agent, input, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=not result.final_output.is_engineering_topic,
    )


guarded_copilot = copilot.clone(
    name="Guarded GitHub Copilot",
    input_guardrails=[engineering_topic_guardrail],
)

In [ ]:
try:
    result = await Runner.run(guarded_copilot, "Write a romantic poem about the moon.")
    print(result.final_output)
except InputGuardrailTripwireTriggered:
    print("Blocked: this is not an engineering request.")

## 3. Output Guardrail

In [ ]:
from agents import OutputGuardrailTripwireTriggered, output_guardrail


class SecretCheck(BaseModel):
    contains_secret: bool
    reasoning: str


secret_guard_agent = Agent(
    name="Secret Output Guard",
    instructions=(
        "Check whether the output includes API keys, passwords, private tokens, or credentials. "
        "Examples include strings starting with sk-, ghp_, xoxb-, or password assignments."
    ),
    model="gpt-4o-mini",
    output_type=SecretCheck,
)


@output_guardrail
async def secret_output_guardrail(ctx: RunContextWrapper[None], agent: Agent, output: str) -> GuardrailFunctionOutput:
    result = await Runner.run(secret_guard_agent, output, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.contains_secret,
    )


safe_copilot = guarded_copilot.clone(
    name="Safe GitHub Copilot",
    output_guardrails=[secret_output_guardrail],
)

## 4. Tracing

In [ ]:
from agents import trace

with trace("github_copilot_mcp_workshop"):
    result = await Runner.run(
        safe_copilot,
        "Use DeepWiki to explain how handoffs work in openai/openai-agents-python. Mention the important modules or classes.",
    )

print(result.final_output)
print("\nTrace dashboard: https://platform.openai.com/traces")

## 5. Try Another Repository

In [ ]:
repo_question = "Use DeepWiki to summarize the architecture of fastapi/fastapi for a backend engineer."

try:
    with trace("github_copilot_second_repo"):
        result = await Runner.run(safe_copilot, repo_question)
    print(result.final_output)
except OutputGuardrailTripwireTriggered:
    print("Blocked: output looked like it contained a secret.")